# Week 2 — Customer Churn Prediction
## Same Training Loop, New Problem

**IIT Madras · Wadhwani School of AI**

---

**Session Plan (60 min):**
1. Data Prep (~10 min) — load, encode, scale
2. Build & Train (~15 min) — nn.Module, BCEWithLogitsLoss, same 5-step loop
3. Evaluate & Debug (~12 min) — confusion matrix, precision/recall, why accuracy lies
4. Week 1 vs Week 2 (~5 min) — side-by-side comparison
5. Q&A (~10 min)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import time

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.10.0+cpu


---
# Part 1: Data Prep (~10 min)

Last week we had images — just `ToTensor()` and go. Real-world tabular data needs more work: categorical encoding, handling bad values, and feature scaling.

### 1.1 Load and Inspect

Telco customer churn dataset: ~7,000 customers, 21 columns. The target is binary — did the customer leave (churn) or not?

In [2]:
# Load dataset
# If running on Colab, upload the CSV or use:
!gdown https://drive.google.com/uc?id=1xM2bFllWdNVVge0awuTKXvY_grM78vXD

df = pd.read_csv('Telco-Customer-Churn.csv')

print(f"Dataset: {df.shape[0]:,} customers, {df.shape[1]} columns")
print(f"\nTarget distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {(df['Churn'] == 'Yes').mean():.1%}  ← imbalanced!")
print(f"\nColumn types:")
print(f"  Numeric:     tenure, MonthlyCharges, TotalCharges")
print(f"  Categorical: everything else (gender, Contract, InternetService, etc.)")

Downloading...
From: https://drive.google.com/uc?id=1xM2bFllWdNVVge0awuTKXvY_grM78vXD
To: /content/Telco-Customer-Churn.csv
100% 970k/970k [00:00<00:00, 12.0MB/s]
Dataset: 7,043 customers, 21 columns

Target distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.5%  ← imbalanced!

Column types:
  Numeric:     tenure, MonthlyCharges, TotalCharges
  Categorical: everything else (gender, Contract, InternetService, etc.)


In [ ]:
# Quick look at a few rows
df.head(3)

### 1.2 Clean and Encode

Neural networks need numbers, not strings. We'll fix the one data quality issue (`TotalCharges` has spaces), encode categoricals, and scale everything.

In [ ]:
# --- Step 1: Fix TotalCharges (has spaces instead of NaN) ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# --- Step 2: Encode target ---
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# --- Step 3: Drop customerID (not a feature) ---
df = df.drop('customerID', axis=1)

# --- Step 4: Encode categoricals with pd.get_dummies ---
# This is the fast approach — one-hot encoding in one line
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=float)

print(f"\nBefore encoding: {df.shape[1]} columns")
print(f"After encoding:  {df_encoded.shape[1]} columns")
print(f"\nAll numeric now: {df_encoded.dtypes.nunique() == 1}")

### 1.3 Split and Scale

Split first, then scale. Scaling after splitting prevents data leakage — the test set shouldn't influence the training normalization.

In [ ]:
# Separate features and target
X = df_encoded.drop('Churn', axis=1).values
y = df_encoded['Churn'].values

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (fit on train only!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit + transform on train
X_test = scaler.transform(X_test)          # only transform on test

print(f"Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} samples")
print(f"\nTrain churn rate: {y_train.mean():.1%}")
print(f"Test churn rate:  {y_test.mean():.1%}")
print("(stratify=y keeps the ratio consistent)")

### 1.4 Convert to PyTorch Tensors

Last step before the model — convert numpy arrays to tensors and set up the DataLoader.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoaders
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

# Handle class imbalance
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)

print(f"Tensor shapes: X={X_train_t.shape}, y={y_train_t.shape}")
print(f"Batches per epoch: {len(train_loader)}")
print(f"\nClass imbalance weight: {pos_weight.item():.2f}")
print(f"  → Missing a churner costs {pos_weight.item():.1f}x more than a false alarm")

---
# Part 2: Build & Train (~15 min)

Same `nn.Module` pattern as week 1's Fashion-MNIST. The architecture is different (tabular input, binary output), but the structure is identical: define layers in `__init__`, wire them in `forward()`.

### 2.1 Model Architecture

New concepts this week: **BatchNorm** (stabilizes training by normalizing activations) and **single output logit** (binary classification). No sigmoid in `forward()` — `BCEWithLogitsLoss` handles it.

In [ ]:
class ChurnPredictor(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.net = nn.Sequential(
            # Layer 1: input → 128
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),       # NEW: normalizes activations
            nn.ReLU(),
            nn.Dropout(0.3),

            # Layer 2: 128 → 64
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Layer 3: 64 → 32
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Output: 32 → 1 raw logit (no sigmoid!)
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)  # squeeze to match target shape

# Create model
input_size = X_train_t.shape[1]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ChurnPredictor(input_size).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Input features: {input_size}")

### 2.2 Loss and Optimizer

Key difference from week 1: `BCEWithLogitsLoss` instead of `CrossEntropyLoss`, and we pass `pos_weight` to handle the 73/27 class imbalance.

In [ ]:
# Loss function — with class imbalance weighting
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

# Optimizer — same Adam as week 1
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 30

print(f"Loss:      BCEWithLogitsLoss (pos_weight={pos_weight.item():.2f})")
print(f"Optimizer: Adam (lr=0.001)")
print(f"Epochs:    {num_epochs}")
print(f"\nWeek 1 used CrossEntropyLoss → multi-class (10 outputs)")
print(f"Week 2 uses BCEWithLogitsLoss → binary (1 output)")

### 2.3 The Training Loop

**Same 5 steps as last week.** Forward → Loss → Zero Grad → Backward → Step. The only structural change is how we compute accuracy (threshold at 0.5 after sigmoid instead of argmax).

In [ ]:
train_losses = []
test_losses = []
test_accs = []

start_time = time.time()

for epoch in range(num_epochs):
    # ===== TRAINING =====
    model.train()
    running_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        pred = model(batch_X)                  # 1. Forward pass
        loss = loss_fn(pred, batch_y)           # 2. Compute loss
        optimizer.zero_grad()                   # 3. Zero gradients
        loss.backward()                         # 4. Backpropagation
        optimizer.step()                        # 5. Update weights

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ===== EVALUATION =====
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            pred = model(batch_X)
            test_loss += loss_fn(pred, batch_y).item()
            # Binary: sigmoid → threshold at 0.5
            predicted = (torch.sigmoid(pred) >= 0.5).float()
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    avg_test_loss = test_loss / len(test_loader)
    accuracy = correct / total
    test_losses.append(avg_test_loss)
    test_accs.append(accuracy)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Test Loss: {avg_test_loss:.4f} | "
              f"Test Acc: {accuracy:.1%}")

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed:.1f}s")
print(f"Final test accuracy: {test_accs[-1]:.1%}")
print(f"\n⚠️  But wait — is accuracy the right metric here?")

---
# Part 3: Evaluate & Debug (~12 min)

Accuracy looks decent, but with 73% non-churners, a model that predicts "no churn" for everyone also gets ~73%. We need better metrics.

### 3.1 Training Curves

First, check if the model is actually learning and if there's overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

epochs_range = range(1, num_epochs + 1)

axes[0].plot(epochs_range, train_losses, '-', label='Train Loss', color='#7C6AEF', linewidth=2)
axes[0].plot(epochs_range, test_losses, '-', label='Test Loss', color='#FB923C', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, [a * 100 for a in test_accs], '-', color='#4ADE80', linewidth=2)
axes[1].axhline(y=73.4, color='#EF4444', linestyle='--', alpha=0.7, label='Always predict "No Churn"')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Test Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("The dashed red line = a model that always says 'No Churn'. We need to beat it meaningfully.")

### 3.2 Confusion Matrix

The confusion matrix shows exactly where the model gets it right and where it fails. This is far more informative than a single accuracy number.

In [ ]:
# Get all predictions
model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.inference_mode():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        logits = model(batch_X)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(batch_y.numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Plot confusion matrix
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['No Churn', 'Churn'])
ax.set_yticklabels(['No Churn', 'Churn'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# Annotate cells
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{labels[i][j]}\n{cm[i, j]}",
                ha='center', va='center', fontsize=14, fontweight='bold',
                color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(f"TN = correctly predicted No Churn")
print(f"TP = correctly predicted Churn (this is what the business cares about!)")
print(f"FN = missed churners (the expensive mistake)")
print(f"FP = false alarms (annoying but cheaper)")

### 3.3 Classification Report

Precision, recall, and F1-score — the metrics that actually matter for imbalanced problems. Recall on class 1 (churn) tells us what fraction of actual churners we caught.

In [ ]:
print("Classification Report:")
print("=" * 55)
print(classification_report(all_labels, all_preds,
                            target_names=['No Churn', 'Churn']))

# AUC score
auc = roc_auc_score(all_labels, all_probs)
print(f"ROC AUC Score: {auc:.4f}")
print(f"\nWhat to look at:")
print(f"  Recall (Churn): What % of actual churners did we catch?")
print(f"  Precision (Churn): Of those we flagged, how many actually churned?")
print(f"  AUC: Overall ranking quality (1.0 = perfect, 0.5 = random)")

### 3.4 ROC Curve

The ROC curve shows the trade-off between catching churners (recall) and false alarms (false positive rate) at every possible threshold.

In [ ]:
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#7C6AEF', linewidth=2.5, label=f'Model (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'r--', alpha=0.5, label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='#7C6AEF')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Higher curve = better model. The dashed red line = guessing randomly.")

### 3.5 Prediction Confidence Distribution

Looking at how confident the model is on churners vs non-churners. A well-calibrated model should give high probabilities to actual churners and low probabilities to non-churners.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(all_probs[all_labels == 0], bins=30, alpha=0.6, label='No Churn (actual)',
        color='#4ADE80', edgecolor='white')
ax.hist(all_probs[all_labels == 1], bins=30, alpha=0.6, label='Churn (actual)',
        color='#EF4444', edgecolor='white')
ax.axvline(x=0.5, color='white', linestyle='--', linewidth=2, label='Threshold = 0.5')

ax.set_xlabel('Predicted Churn Probability', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Ideal: green (no churn) all on the left, red (churn) all on the right.")
print("Overlap in the middle = the cases the model is uncertain about.")

---
# Part 4: Week 1 vs Week 2 (~5 min)

Let's put the two sessions side-by-side to see what changed and — more importantly — what stayed the same.

In [ ]:
comparison = {
    '': ['Week 1: Fashion-MNIST', 'Week 2: Customer Churn'],
    'Task': ['Multi-class (10 classes)', 'Binary (churn/no churn)'],
    'Input type': ['28×28 grayscale images', 'Tabular: 21 mixed columns'],
    'Data prep': ['ToTensor + Normalize', 'Encoding + scaling + imputation'],
    'Features': ['784 pixels (raw)', f'{input_size} engineered features'],
    'Architecture': ['784→128→64→10', f'{input_size}→128→64→32→1'],
    'Loss function': ['nn.CrossEntropyLoss()', 'nn.BCEWithLogitsLoss(pos_weight)'],
    'Output': ['10 logits → argmax', '1 logit → sigmoid → threshold'],
    'Key metric': ['Accuracy (balanced classes)', 'Recall + AUC (imbalanced)'],
    'New concepts': ['Flatten, DataLoader', 'BatchNorm, pos_weight, confusion matrix'],
}

print("=" * 72)
print(f"{'':20s} | {'WEEK 1':24s} | {'WEEK 2':24s}")
print("=" * 72)
for key, (v1, v2) in comparison.items():
    if key == '':
        continue
    print(f"{key:20s} | {v1:24s} | {v2:24s}")
print("=" * 72)

print("\n✅ What DIDN'T change:")
print("   • nn.Module: define layers in __init__, wire in forward()")
print("   • The 5-step training loop: forward → loss → zero_grad → backward → step")
print("   • model.train() for training, model.eval() + inference_mode() for testing")
print("   • Adam optimizer as the default")
print("   • Output raw logits, let the loss handle the activation")
print("\n   This pattern works for ANY neural network task.")

### Model Saving

In [ ]:
torch.save(model.state_dict(), 'churn_model.pth')
print("Model saved to churn_model.pth")

---
# Recap

### 1. The 5-Step Loop is Universal
```python
pred = model(X)           # Forward
loss = loss_fn(pred, y)   # Loss
optimizer.zero_grad()     # Zero gradients
loss.backward()           # Backprop
optimizer.step()          # Update
```
Images, tabular, text, audio — same loop.

### 2. Match Loss to Task
- Multi-class → `CrossEntropyLoss` (N outputs)
- Binary → `BCEWithLogitsLoss` (1 output)
- Regression → `MSELoss` (1 output, no activation)

### 3. Real-World Data Needs More Care
- Encoding categoricals, scaling numerics, handling missing values
- Class imbalance → weighted loss + metrics beyond accuracy
- Confusion matrix, precision, recall, AUC > accuracy

---
*Questions?*